# Libraries

In [20]:
import pandas as pd
import numpy as np

# Theory

## Y-axis Explanation

The authoritarian-libertarian axis typically contrasts:

- Authoritarian (top/Y-positive): Emphasis on strong state authority, traditional values, collectivism, outcome-oriented justice, and social responsibilities over individual rights.
- Libertarian (bottom/Y-negative): Emphasis on individual freedoms, minimal state interference, secularism, individualism, procedural justice, and personal rights.

## Proxy

no direct dataset labelling y-axis, auth-libertarian, so use proxy

Dataset: https://github.com/LST1836/MITweet/blob/main/data/MITweet.csv


Selected facets for proxy y-axis:
- I5 (Ethical Pursuit, EP):
    - Left (0: Ethical Liberalism, e.g., support for sexual liberation/abortion) ≈ libertarian
    - Right (2: Ethical Conservatism, e.g., moral/religious restrictions) ≈ authoritarian
- I6 (Church-State Relations, CSR): 
    - Left (0: Secularism, separation of church/state) ≈ libertarian
    - Right (2: Caesaropapism, unified church/state) ≈ authoritarian
- I7 (Cultural Value, CV):
    - Left (0: Collectivism, group over individual) ≈ authoritarian
    - Right (2: Individualism, human-centered values) ≈ libertarian. (Note: Flip the mapping here.)
- I11 (Justice Orientation, JO): 
    - Left (0: Result Justice, state-enforced fair outcomes) ≈ authoritarian
    - Right (2: Procedural Justice, fair processes/rights) ≈ libertarian. (Flip mapping.)
- I12 (Personal Right, PeR):
    - Left (0: Social Responsibility, duties to society) ≈ authoritarian
    - Right (2: Individual Right, protecting individual freedoms) ≈ libertarian. (Flip mapping.)

I1 ~ I12 : ideology labels for the 12 facets. 0 , 1 , 2 mean left-leaning, center, right-leaning, respectively. -1 means "Unrelated", so no ideology label

In [21]:
# Define the relevant facets for Y-axis proxy (authoritarian-libertarian)
facets = {
    'I5': {'flip': False},  # Ethical Pursuit: left=libertarian, right=authoritarian
    'I6': {'flip': False},  # Church-State Relations: left=libertarian, right=authoritarian
    'I7': {'flip': True},   # Cultural Value: left=authoritarian, right=libertarian (flip)
    'I11': {'flip': True},  # Justice Orientation: left=authoritarian, right=libertarian (flip)
    'I12': {'flip': True}   # Personal Right: left=authoritarian, right=libertarian (flip)
}

In [22]:
df = pd.read_csv('https://raw.githubusercontent.com/LST1836/MITweet/main/data/MITweet.csv')

In [23]:
df.head()

,topic,tweet,tokenized tweet,R1,R2,R3,R4,R5,R1-1-1,R2-1-2,...,I3,I4,I5,I6,I7,I8,I9,I10,I11,I12
0,Inflation Reduction Act,The New Methane Emissions Charge: One (Limited...,The New Methane Emissions Charge : One ( Limit...,0,1,0,0,0,0,0,...,0,-1,-1,-1,-1,-1,-1,-1,-1,-1
1,Inflation Reduction Act,Forget about protecting the people and communi...,Forget about protecting the people and communi...,0,0,1,0,0,0,0,...,-1,-1,-1,-1,2,-1,-1,-1,-1,-1
2,Inflation Reduction Act,Curious to know how much money will be availab...,Curious to know how much money will be availab...,0,1,0,0,0,0,0,...,0,0,-1,-1,-1,-1,-1,-1,-1,-1
3,Inflation Reduction Act,"""With #inflation finally starting to moderate,...",""" With #inflation finally starting to moderate...",0,1,0,0,0,0,0,...,1,1,-1,-1,-1,-1,-1,-1,-1,-1
4,Inflation Reduction Act,What do we want? To #BuildBackForJustice! When...,What do we want ? To #BuildBackForJustice ! Wh...,1,1,0,0,1,1,1,...,-1,0,-1,-1,-1,-1,-1,0,-1,-1


In [40]:
len(df)

5065

In [24]:
# Function to remap ideology labels to Y-score (-1 lib to +1 auth)
def remap_label(label, flip):
    if label == -1:
        return np.nan  # Unrelated: skip
    mapping = {0: -1, 1: 0, 2: 1}  # left=-1 (lib), center=0, right=+1 (auth)
    if flip:
        mapping = {0: 1, 1: 0, 2: -1}  # Invert for auth-left facets
    return mapping[label]

__EXPLANATION__:
Defines a function that takes a label (from I columns) and a flip flag, then remaps it to a Y-score:
- If label == -1 (unrelated), returns NaN (skippable).
- Default mapping: 0 (left) → -1 (lib), 1 (center) → 0, 2 (right) → +1 (auth).
- If flip=True, inverts: 0 → +1 (auth), 1 → 0, 2 → -1 (lib). ensures consistent scoring across facets.

In [25]:
# Compute Y-proxy score for each tweet (average of remapped scores)
y_scores = []
for _, row in df.iterrows():
    scores = [remap_label(row[col], info['flip']) for col, info in facets.items() if not np.isnan(remap_label(row[col], info['flip']))]
    y_proxy = np.mean(scores) if scores else np.nan
    y_scores.append(y_proxy)

df['y_proxy'] = y_scores

__EXPLANATION__:
Loops over each row in df:
- For each selected facet, calls remap_label and collects non-NaN scores in a list.
- Computes the mean of those scores (y_proxy); if no valid scores, sets NaN.
- Appends to y_scores list, then adds it as a new 'y_proxy' column in df.
- This creates the continuous proxy score per tweet (e.g., -0.5 for moderate libertarian lean)

In [35]:
df['y_proxy'].unique()

array([-1.        ,  1.        ,  0.5       , -0.5       , -0.33333333,
       -0.66666667,  0.33333333, -0.25      ,  0.66666667])

In [26]:
df['y_proxy'].head()

0    NaN
1   -1.0
2    NaN
3    NaN
4    NaN
Name: y_proxy, dtype: float64

In [27]:
# Drop NaNs and exact centers (y_proxy == 0) for binary labels
df = df.dropna(subset=['y_proxy'])
df = df[df['y_proxy'] != 0]

In [28]:
df['y_proxy'].head()

1    -1.0
5     1.0
6    -1.0
12    1.0
13   -1.0
Name: y_proxy, dtype: float64

In [29]:
# Change 0 = authoritarian (y_proxy > 0), 1 = libertarian (y_proxy < 0)
df['label'] = np.where(df['y_proxy'] > 0, 0, 1)

In [30]:
# Select only tweet text and label
output_df = df[['tweet', 'label']]

In [31]:
# 0 = authoritarian, 1 = libertarian
output_df.head()

,tweet,label
1,Forget about protecting the people and communi...,1
5,Are you with an #environmentalnonprofit leadin...,0
6,Curious if #IRA can help you electrify your ho...,1
12,Learn more from \n on how huge investments fro...,0
13,Almost a year since this Vox video on grid cha...,1


In [41]:
len(output_df)

5065

In [32]:
output_df.to_csv('MITweet_Auth_Lib.csv', index=False)